# MINT-TTS — Egyptian Arabic homograph experiment

**The question.** Egyptian Arabic is written without short vowels, so the same
spelling carries several pronunciations and only context decides which:

| written | reading A | reading B |
|---|---|---|
| `علم` | `عَلَم` *3alam* — flag | `عِلْم` *3elm* — science |
| `عمرك` | `عُمرَك` *3omrak* — to a man | `عُمرِك` *3omrik* — to a woman |
| `ضرب` | `ضَرَب` — he hit | `ضُرِب` — he was hit |

Most Arabic TTS sidesteps this by demanding diacritised input, which moves the
problem to whoever types the text. This experiment asks whether the model can
resolve it from context alone, and whether it spends **more computation** on
the words that need it.

## No word lists anywhere

There is **no hand-written homograph list** in this repository, and no
hand-written probe sentences. Both would be the wrong design:

* a list of a few dozen words covers nothing of a 62k-word vocabulary;
* it encodes the author's guesses rather than the corpus (an early draft
  marked `مصر` ambiguous — in a corpus of history programmes it is "Egypt"
  every single time);
* it would have to be rewritten by hand for every new dialect or language.

Instead, **ambiguity is measured**. A homograph is a spelling whose
pronunciation varies with context, and both halves are observable: the mel
frames the aligner assigns to a word, and the frozen LM's vector for that
occurrence. `scripts/mine_ambiguity.py` clusters each word's pronunciations
and asks whether context predicts the cluster. A word scores high only when it
is said in genuinely different ways **and** the surrounding words say which
way applies.

The probe sentences are mined from the corpus the same way, so they are
in-domain by construction rather than sentences somebody imagined.

**Three components:**

1. **`ar_char` frontend** — raw graphemes, so the ambiguity actually reaches
   the model. (On English, the `ipa` frontend pre-resolved every homograph and
   the experiment measured nothing. That null result is why this matters.)
2. **Frozen MARBERTv2** — one contextual vector per word. Character statistics
   over 68 hours cannot recover lexical semantics; a dialect-pretrained LM
   already has them. Cached offline, so training never runs BERT.
3. **Difficulty-aware ACT routing** — a measured-ambiguous token pays ~25% of
   the per-step compute price, so depth where it is needed is affordable while
   easy words stay under pressure.

---
## 0. Environment

Everything expensive is kept on **Google Drive** under `MyDrive/Mint-TTS/egyptian`,
so a new session costs a few minutes of copying instead of hours of
downloading and preprocessing:

| what | where on Drive | restored by |
|---|---|---|
| preprocessed corpus (mels, pitch, energy, MARBERT vectors) | `data/preprocessed_egyptian.tar` | section 2 |
| raw wavs (only needed to re-preprocess) | `data/raw_egyptian.tar` | section 2, only if needed |
| mined ambiguity tables | `data/ambiguity/` | section 2 and after every mining pass |
| checkpoints + TensorBoard logs | `runs/` (the repo's `runs/` is a link to it) | automatically |
| MARBERT + BigVGAN weights | `hf_cache/` | automatically |

Training always runs with `--resume auto`: it starts from scratch when there is
no checkpoint, and otherwise continues from the most advanced one (`last.pt` is
written every 1000 steps, so a disconnect loses at most that).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/Mint-TTS/egyptian')
DRIVE_DATA = DRIVE / 'data'
DRIVE_RUNS = DRIVE / 'runs'
for d in (DRIVE_DATA / 'ambiguity', DRIVE_RUNS):
    d.mkdir(parents=True, exist_ok=True)

# Model downloads (MARBERT, BigVGAN) are cached on Drive too. Must be set
# before anything imports huggingface_hub; `!python` subprocesses inherit it.
os.environ['HF_HOME'] = '/content/drive/MyDrive/Mint-TTS/hf_cache'
print('Drive state ->', DRIVE)

In [ ]:
import sys

REPO = Path('/content/MINT-TTS')
if not REPO.exists():
    !git clone https://github.com/MohammedAly22/MINT-TTS.git {REPO}
else:
    !git -C {REPO} pull --ff-only
os.chdir(REPO)
sys.path.insert(0, str(REPO))

# Checkpoints and TensorBoard logs go straight to Drive.
runs = REPO / 'runs'
if runs.exists() and not runs.is_symlink():
    !cp -rn {runs}/. {DRIVE_RUNS}/ && rm -rf {runs}
if not runs.exists():
    runs.symlink_to(DRIVE_RUNS, target_is_directory=True)
print('cwd:', Path.cwd(), '| runs ->', os.path.realpath(runs))

In [ ]:
!pip install -q -r requirements.txt
# Arabic extras: MARBERT, the corpus loader, BigVGAN's mel helpers (librosa),
# and correct Arabic in figures.
!pip install -q transformers soundfile librosa arabic-reshaper python-bidi

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

---
## 1. What the frontend does to Arabic

Check the text pipeline before training. If the normaliser stripped or folded
the undiacritised spellings, the experiment would be impossible and the
failure would be invisible later.

In [ ]:
from mint_tts.text.arabic import ArabicNormalizer, ARABIC_WORD_RE

norm = ArabicNormalizer()
examples = [
    "انا رسمت علم مصر",
    "انا بحب العلم جدا و نفسي ابقا عالم",
    "عمرك فكرتي الراجل بتاع غزل البنات؟",
    "عندي ٢٥ كتاب و ٥٠٪ منهم عربي",
    "قال ليـــ إزيك يا آدم؟",
    "عَلَم مُشَكَّل",
]
for raw in examples:
    print(f"in  : {raw}")
    print(f"out : {norm(raw)}\n")

### Numbers must be Egyptian, not MSA

`num2words(lang="ar")` emits Modern Standard Arabic in the **nominative**:
`عشرون` where every speaker in this corpus says `عشرين`. That is not cosmetic
— the transcript is what the aligner maps onto the audio, so an MSA number
word is a text/audio mismatch the model would be trained on. `text/numbers_ar.py`
handles the case, the dual, and the 3-10 plural agreement.

In [ ]:
from mint_tts.text.numbers_ar import number_to_words

for v in [2, 3, 11, 20, 25, 100, 1000, 2000, 3000, 1984]:
    print(f"{v:>6} -> {number_to_words(v)}")
print()
print("2 -> اتنين (not اثنان) | 20 -> عشرين (not عشرون) | 100 -> مية (not مائة)")

### Punctuation reaches the model

The Arabic `؟` becomes `?` and survives as a **token**, which is what carries
question intonation. It does not appear in a *word* list, because a word regex
matches words — check the tokens, not `ARABIC_WORD_RE`.

In [ ]:
from mint_tts.config import Config
from mint_tts.text.tokenizer import build_text_processor

tp = build_text_processor(Config({"text": {
    "input_type": "ar_char", "lowercase": True, "add_bos_eos": True,
    "add_word_boundary": True, "add_punctuation": True,
    "allow_vocab_growth": True, "arabic": {}}}))

enc = tp.encode("عامل ايه النهاردة؟")
print("tokens :", " ".join(enc.tokens))
print("words  :", enc.words)
print("has ?  :", "?" in enc.tokens)

### Does MARBERT separate the readings?

The load-bearing assumption. If the LM gives `علم` the same representation in
both senses, nothing downstream can disambiguate it.

**Raw cosine is the wrong test**, for two reasons, and getting this wrong will
make a working model look broken:

* **Anisotropy.** BERT vectors occupy a narrow cone, so cosine between *any*
  two words in the last layer runs 0.95–0.99. On this model two completely
  **unrelated** words score **0.979** — so a homograph pair at 0.984 is not
  evidence of failure, it is evidence that the metric is saturated.
* **Wrong task.** Neither the adapter nor the miner uses raw cosine. The
  adapter learns a projection; the miner runs a leave-one-out classifier.
  Both exploit a consistent *direction* that cosine barely registers.

So this cell measures what is actually used: centred cosine (anisotropy
removed) and leave-one-out classification against a shuffled control.

In [ ]:
from mint_tts.modules.semantic import SemanticEncoder
import numpy as np

LAYER = -3          # see configs/egyptian_homograph.yaml for why not -1
enc_lm = SemanticEncoder("marbert", layer=LAYER,
                         device="cuda" if torch.cuda.is_available() else "cpu")

ALAM = "علم"
FLAG = [                                   # 3alam = flag
    "انا رسمت علم مصر",
    "رفعوا علم البلاد فوق المبني",
    "علم احمر و ابيض و اسود",
    "اللاعب لف علم ناديه حوالينه",
    "حطوا علم كبير علي السطح",
    "العلم بيرفرف في الهوا",
]
SCIENCE = [                                # 3elm = science
    "درست علم الاحياء في الجامعة",
    "علم النفس موضوع صعب",
    "العلم نور و الجهل ظلام",
    "هو بيدرس علم الفلك",
    "العلم بيتقدم كل يوم",
    "مافيش حاجة اسمها علم سهل",
]

def vec(sentence):
    """The vector for 3alam/3elm in this sentence (bare or with 'al-')."""
    words = ARABIC_WORD_RE.findall(norm(sentence))
    hit = next((w for w in words if w == ALAM or w == "ال" + ALAM), None)
    return enc_lm.encode(words).vectors[words.index(hit)] if hit else None

vf = [v for s in FLAG if (v := vec(s)) is not None]
vs = [v for s in SCIENCE if (v := vec(s)) is not None]
X = np.stack(vf + vs)
y = np.array([0] * len(vf) + [1] * len(vs))

def loo_accuracy(X, y):
    """Leave-one-out nearest centroid -- exactly what the miner computes."""
    U = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)
    ok = 0
    for i in range(len(U)):
        keep = np.ones(len(U), bool); keep[i] = False
        c = [U[keep & (y == k)].mean(0) for k in (0, 1)]
        c = [v / max(np.linalg.norm(v), 1e-9) for v in c]
        ok += int(int(U[i] @ c[1] > U[i] @ c[0]) == y[i])
    return ok / len(U)

rng = np.random.default_rng(0)
acc = loo_accuracy(X, y)
shuffled = np.mean([loo_accuracy(X, rng.permutation(y)) for _ in range(20)])

Xc = X - X.mean(0, keepdims=True)          # remove the anisotropy cone
U = Xc / np.maximum(np.linalg.norm(Xc, axis=1, keepdims=True), 1e-9)
S = U @ U.T
same = np.mean([S[i, j] for i in range(len(X)) for j in range(i + 1, len(X)) if y[i] == y[j]])
diff = np.mean([S[i, j] for i in range(len(X)) for j in range(len(X)) if y[i] != y[j]])

print(f"centred cosine, same reading  : {same:+.3f}")
print(f"centred cosine, across reading: {diff:+.3f}   <- must be clearly LOWER")
print()
print(f"leave-one-out accuracy        : {acc:.2f}")
print(f"  same test, shuffled labels  : {shuffled:.2f}   <- the chance baseline")
print()
if acc > 0.75 and acc - shuffled > 0.25:
    print("PASS -- the readings are separable. This is the signal the miner")
    print("and the adapter use; proceed to training.")
else:
    print("WEAK -- try another LAYER (-2, -4) or another model before")
    print("spending GPU hours. Do NOT judge this by raw cosine.")

---
## 2. The corpus

15,653 clips, one speaker, 24 kHz, undiacritised and unpunctuated. The official
splits are disjoint **by source video** and `prepare_egyptian.py` honours them
— a random split would put near-duplicates of training clips into validation
and make every validation number optimistic.

**Restore first.** The next cell looks for the preprocessed corpus on Drive and
unpacks it to the local disk (training reads ~60k small files per epoch; that
must be the local SSD, never Drive). Only when nothing usable is on Drive do the
download and preprocessing cells below do any work — they skip themselves
otherwise.

The archive is checked against the config: the mel settings changed to **100
bands** so that the BigVGAN vocoder applies (section 5), and an older 80-band
archive is detected and re-preprocessed. Its MARBERT vectors are reused, so that
costs only the audio pass.

In [ ]:
import json, shutil
import numpy as np
from mint_tts.config import load_config

CONFIG = 'configs/egyptian_homograph.yaml'
cfg = load_config(CONFIG)
PRE = Path(cfg.data.preprocessed_dir)
PRE_TAR = DRIVE_DATA / 'preprocessed_egyptian.tar'
RAW_TAR = DRIVE_DATA / 'raw_egyptian.tar'

def restore_ambiguity():
    for f in (DRIVE_DATA / 'ambiguity').glob('ambiguity*.json'):
        shutil.copy(f, PRE / f.name)

def save_ambiguity():
    for f in PRE.glob('ambiguity*.json'):
        shutil.copy(f, DRIVE_DATA / 'ambiguity' / f.name)
    print('ambiguity tables saved to Drive:', sorted(p.name for p in PRE.glob('ambiguity*.json')))

def preprocessed_ok():
    idx = PRE / 'train.jsonl'
    if not idx.exists():
        return False
    row = json.loads(idx.open(encoding='utf-8').readline())
    n_mels = np.load(row['mel'], mmap_mode='r').shape[0]
    if n_mels != cfg.audio.n_mels:
        print(f'cached mels have {n_mels} bands, config wants {cfg.audio.n_mels}: re-preprocessing')
        return False
    return True

if not (PRE / 'train.jsonl').exists() and PRE_TAR.exists():
    print('restoring the preprocessed corpus from Drive ...')
    !tar -xf {PRE_TAR} -C {REPO}
NEED_PREPROCESS = not preprocessed_ok()
if PRE.exists():
    restore_ambiguity()
print('NEED_PREPROCESS =', NEED_PREPROCESS)

In [ ]:
# Raw audio: only needed when preprocessing has to run.
# The download is ~12 GB of individual wavs; lower --workers if the Hub
# rate-limits you, and --skip-existing resumes a partial download.
SAVE_RAW_TO_DRIVE = True     # ~12 GB on Drive, saves the slow re-download

if NEED_PREPROCESS:
    if not Path('data/egyptian').exists() and RAW_TAR.exists():
        print('restoring raw wavs from Drive ...')
        !tar -xf {RAW_TAR} -C {REPO}
    if not Path('data/egyptian').exists():
        !python scripts/prepare_egyptian.py --out data/egyptian --workers 8
        if SAVE_RAW_TO_DRIVE:
            !tar -cf {RAW_TAR}.tmp data/egyptian filelists/egyptian_* && mv {RAW_TAR}.tmp {RAW_TAR}
else:
    print('preprocessed corpus present -- no raw audio needed')

---
## 3. Preprocessing

Mel/pitch/energy, cached tokenisation, **and** the frozen MARBERT vectors.
Caching the LM output here is what keeps training fast: a 163M-parameter BERT
forward pass per step would otherwise dominate an ~18M-parameter acoustic
model. The cache is keyed by a hash of `(model, layer)`, so switching LM cannot
silently reuse stale vectors — and vectors already on disk are reused, so
re-preprocessing for new mel settings only redoes the audio.

Mels are stored as float16 (`audio.mel_dtype`), which halves the archive.
When this finishes, the whole corpus is archived to Drive.

In [ ]:
if NEED_PREPROCESS:
    !python scripts/preprocess.py --config {CONFIG} --workers 8
    print('archiving the preprocessed corpus to Drive ...')
    !tar -cf {PRE_TAR}.tmp {PRE.as_posix()} && mv {PRE_TAR}.tmp {PRE_TAR}
    !ls -lh {PRE_TAR}
    restore_ambiguity()
else:
    print('skipped: preprocessed corpus restored from Drive')

In [ ]:
stats = json.loads((PRE / 'stats.json').read_text(encoding='utf-8'))
for k in ['n_utterances', 'total_hours', 'vocab_size', 'input_type',
          'semantic_model', 'semantic_hidden_size', 'semantic_files']:
    print(f'{k:22s} {stats.get(k)}')

---
## 4. Discover which words are ambiguous

**This is the step that replaces a hand-written homograph list.**

Run it text-only now (it needs only the cached LM vectors), then again after
training with `--checkpoint`, when the aligner can say which mel frames belong
to which word. The acoustic pass is much stronger evidence: it measures actual
*pronunciation* variation rather than *meaning* variation.

    preprocess → mine (text-only) → train → re-mine (acoustic) → train on

Each mode writes its own file (`ambiguity.text_only.json`,
`ambiguity.acoustic.json`); `ambiguity.json` is the **active** table the trainer
reads. The text-only pass is skipped when a table was already restored from
Drive.

In [ ]:
if not (PRE / 'ambiguity.json').exists():
    !python scripts/mine_ambiguity.py --config {CONFIG} --top 40
    save_ambiguity()
else:
    print('active ambiguity table restored from Drive -- delete it to re-mine')

Look at that list. These words were discovered from the corpus — nobody told
the system about any of them. They should be spellings whose reading genuinely
depends on context. Some will be words you would not have thought to list;
that is the point.

If the list looks like noise, the text-only signal is weak on this corpus;
proceed to training anyway and re-mine acoustically afterwards.

In [ ]:
from mint_tts.text.ambiguity import AmbiguityTable

table = AmbiguityTable.load("data/preprocessed/egyptian/ambiguity.json")
print(f"{len(table)} word types scored\n")
print("most ambiguous:")
for w, s in table.top(15):
    print(f"  {w:<16} {s:.3f}")
print("\nsome specific words:")
for w in ["علم", "مصر", "كتب", "عمرك", "النهاردة"]:
    print(f"  {w:<16} {table.score(w):.3f}")

---
## 5. A real vocoder

Griffin-Lim is too rough to judge a homograph by ear. The vocoder is **frozen
and shared** across every run, so a quality difference between two
experiments can never come from it.

It is NVIDIA **BigVGAN-v2 24 kHz**, whose mel is *exactly* this corpus's
preprocessing at 100 bands (slaney filters, reflect padding, `center=False`,
log clamp 1e-5, fmax 12 kHz), so its pretrained weights apply unchanged — the
trainer refuses a vocoder whose mel settings differ. (`nvidia/tts_hifigan` is a
22 kHz NeMo archive with no `model.ckpt`, which is why that download 404'd.)
The weights are cached on Drive after the first fetch.

In [ ]:
!python scripts/download_vocoder.py --bigvgan || \
  echo "Fetch failed -- training still works, but audio will be Griffin-Lim."

---
## 6. Train

Watch these in order. Each can invalidate everything below it:

| metric | meaning | act if |
|---|---|---|
| `align/entropy_ratio` | aligner health | still > 0.5 at 8k steps → stop, nothing downstream is meaningful |
| `val/mcd_vs_chance` | is the audio utterance-specific? | ≥ 1.0 → the model says the same thing regardless of input |
| `semantic/delta_norm` | is the LM path being used? | stays 0 → semantics are dead, results come from elsewhere |
| `compute/difficulty_contrast` | **the claim** | should rise above 0 after warmup |
| `compute/encoder_depth_spread` | is the router differentiating? | ~0 → collapsed to a constant |
| `homograph/divergence_ratio` | do readings differ? | ~1 → same pronunciation in both contexts |
| `probe/length_corr` | the trivial solution | ~1 → router only learned sentence length |

The trainer prints at step 0 whether difficulty came from the **mined** table
or the weaker **structural** bootstrap, and warns if the probe sets are empty
because mining has not run.

Compute pressure starts at step 12,000 (`loss.compute.warmup_steps`). Before
then the router is held at full depth on purpose — an unpenalised router
collapses to a constant and takes alignment with it — so
`difficulty_contrast` and `depth_spread` read **0.0 until then**. That is
correct, not a failure.

**Speed.** `train/steps_per_sec` is logged. The per-step costs that used to
dominate on a large GPU are gone: the alignment prior is built on the GPU
instead of in the DataLoader, monotonic alignment search is compiled (numba),
diagnostics only run on logged steps, and bf16 runs without a loss scaler. The
first logged window includes worker start-up and numba compilation, so judge
the rate from the second one.

TensorBoard opens in its **own browser tab** — click the link the next cell
prints.

In [ ]:
import subprocess, time
subprocess.Popen(['tensorboard', '--logdir', str(DRIVE_RUNS), '--port', '6006',
                  '--bind_all', '--reload_multifile', 'true'])
time.sleep(6)
from google.colab import output
output.serve_kernel_port_as_window(6006)   # <- click the link that appears

In [ ]:
# Fresh run or continuation -- the same command either way.
!python scripts/train.py --config {CONFIG} --resume auto

### Re-mine with acoustics, then continue

Once `align/entropy_ratio` is well below 0.3, the aligner can locate each
word's frames and the mining pass can measure *pronunciation* variation
directly. This usually changes the table substantially.

Mining **too early** gives noise: at step 2,000 the word spans are mostly
wrong, every word scores near zero, and the table fills with arbitrary words.
The script therefore measures the alignment entropy itself and only promotes
the acoustic table to `ambiguity.json` when it is at or below 0.3; otherwise it
keeps the result in `ambiguity.acoustic.json` and leaves the active table
alone. It runs batched on the GPU, a few minutes for the whole corpus.

In [ ]:
from mint_tts.utils.common import find_latest_checkpoint

CKPT = find_latest_checkpoint(Path('runs/egyptian_homograph/checkpoints'))
print('mining with', CKPT)
!python scripts/mine_ambiguity.py --config {CONFIG} --checkpoint {CKPT} --top 40
save_ambiguity()

# Then continue training; the dataset picks up the new scores on restart.
# !python scripts/train.py --config {CONFIG} --resume auto

---
## 7. Listen

The automatic metric says whether two renderings **differ**. It cannot say
whether they differ *correctly* — only a speaker can.

In [ ]:
from mint_tts.inference.synthesize import Synthesizer
from IPython.display import Audio, display

syn = Synthesizer.from_checkpoint("runs/egyptian_homograph/checkpoints/best.pt")

pairs = [
    ("علم — flag",        "انا رسمت علم مصر"),
    ("علم — science",     "انا بحب العلم جدا و نفسي ابقا عالم لما اكبر"),
    ("عمرك — to a man",   "يا عم عمرك شفت حاجة زي كده يا راجل"),
    ("عمرك — to a woman", "عمرك فكرتي الراجل بتاع غزل البنات بينفخ الكيس ازاي"),
]
for label, text in pairs:
    res = syn(text, quality=0.9)
    print(f"--- {label}")
    print(res.summary())
    display(Audio(res.wav.cpu().numpy(), rate=res.sample_rate))

### Where did the compute go?

Per-word depth, with the words the *mining pass* found ambiguous highlighted.
If the hypothesis holds they sit above the rest, and function words sit at the
floor.

In [ ]:
import matplotlib.pyplot as plt

text = "عمرك فكرتي الراجل بتاع غزل البنات بينفخ الكيس ازاي"
res = syn(text, quality=0.9)
words, depth = res.encoded.words, res.word_complexity
amb = [table.score(w) for w in words]          # MINED, not a word list
colors = ["#d9534f" if a > 0.2 else "#5b8def" for a in amb]

try:
    import arabic_reshaper
    from bidi.algorithm import get_display
    labels = [get_display(arabic_reshaper.reshape(w)) for w in words]
except ImportError:
    labels = words

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.bar(range(len(words)), depth, color=colors)
ax.set_xticks(range(len(words)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=12)
ax.set_ylabel("encoder depth (fraction of max)")
ax.set_title("red = measured ambiguous (mined from the corpus)")
ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

for w, d, a in zip(words, depth, amb):
    print(f"  {w:>12s}  depth={d:.3f}  mined_ambiguity={a:.2f}")

### The speed claim

Easy sentences should be cheaper *and* faster than hard ones. If they cost the
same, the router is not allocating.

In [ ]:
import time

tests = [
    ("easy", "عامل ايه النهاردة؟"),
    ("long but easy", "انا رحت السوق و اشتريت عيش و لبن و جبنة و زيتون و رجعت البيت"),
    ("hard", "عمرك فكرتي الراجل بتاع غزل البنات بينفخ الكيس ازاي؟ هسيبك تجاوبي و تخمني"),
]
print(f"{'sentence':<18s}{'words':>6s}{'depth':>8s}{'RTF':>9s}{'saving':>9s}")
for label, text in tests:
    syn(text, quality=0.9)                         # warm up
    t0 = time.perf_counter()
    r = syn(text, quality=0.9)
    dt = time.perf_counter() - t0
    audio_s = r.mel.shape[-1] * syn.cfg.audio.hop_length / syn.cfg.audio.sample_rate
    print(f"{label:<18s}{len(r.encoded.words):>6d}{r.token_complexity.mean():>8.3f}"
          f"{dt/max(audio_s,1e-6):>9.4f}{r.flops.saving*100:>8.1f}%")

### The quality/compute curve

One checkpoint spans the whole trade-off, because `q` was sampled during
training.

In [ ]:
text = "انا بحب العلم جدا و نفسي ابقا عالم لما اكبر"
for q in [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    r = syn(text, quality=q)
    print(f"q={q:<4.1f} depth={r.token_complexity.mean():.3f} "
          f"FLOPs={r.flops.total:.2e} saving={r.flops.saving*100:5.1f}%")
    display(Audio(r.wav.cpu().numpy(), rate=r.sample_rate))

---
## 8. The controls

A positive result on the main run means little alone. These are cheap —
preprocessing and mining are shared, only training re-runs.

**`egyptian_nosemantic`** — identical except MARBERT is off. If this also
separates the readings, the character encoder was sufficient and the LM is
dead weight. If it flattens, the semantic path is doing the work.

**`egyptian_dense`** — no routing at all. Gives the quality ceiling, and shows
whether disambiguation needs adaptive depth or just the LM features.

In [ ]:
!python scripts/train.py --config configs/egyptian_nosemantic.yaml --resume auto
# !python scripts/train.py --config configs/egyptian_dense.yaml --resume auto

In [ ]:
import glob
from tensorboard.backend.event_processing import event_accumulator

def final(run, tag):
    files = glob.glob(f"runs/{run}/**/events.out.tfevents.*", recursive=True)
    if not files:
        return None
    ea = event_accumulator.EventAccumulator(max(files)); ea.Reload()
    if tag not in ea.Tags().get("scalars", []):
        return None
    return ea.Scalars(tag)[-1].value

TAGS = ["homograph/divergence_ratio", "compute/difficulty_contrast",
        "probe/contrast", "probe/length_corr", "val/mcd", "semantic/delta_norm"]
runs = ["egyptian_homograph", "egyptian_nosemantic", "egyptian_dense"]

print(f"{'metric':<34s}" + "".join(f"{r[9:]:>16s}" for r in runs))
for tag in TAGS:
    row = "".join(f"{v:>16.4f}" if (v := final(r, tag)) is not None else f"{'-':>16s}"
                  for r in runs)
    print(f"{tag:<34s}{row}")

---
## 9. How to read the outcome

**Positive if,** on `egyptian_homograph`:

- `homograph/divergence_ratio` comfortably above 1 (readings differ more than
  ordinary contextual variation), **and**
- `compute/difficulty_contrast` above 0 (measured-ambiguous tokens get more
  depth), **and**
- `probe/length_corr` *not* near 1 (it is not just sentence length), **and**
- `egyptian_nosemantic` is clearly weaker on the first two.

**A caution on `difficulty_contrast`.** The compute penalty *deliberately*
makes depth cheap on ambiguous tokens, so a positive contrast is partly by
construction. It is evidence that the mechanism works, not that the model
discovered ambiguity by itself. The load-bearing comparisons are
`divergence_ratio` (did the pronunciation actually change?) and the
no-semantics control.

**And then you still listen.** Differing is not differing *correctly* — a
model could render both readings wrongly but differently and score well. Every
automatic number here is necessary; none is sufficient.

**If `semantic/delta_norm` stayed at 0**, the semantic path never escaped its
zero initialisation and every homograph number comes from somewhere else.
Check that preprocessing wrote the vectors and the hidden size matches.

**If the mined list looks like nonsense**, the acoustic evidence is not clean
enough yet — usually alignment. Re-mine after `align/entropy_ratio` drops
below 0.3.